In [13]:
#from langchain_core.tools import tool
from langchain.tools import tool
from langchain_ollama import ChatOllama

In [14]:
@tool
def multiply(a:int,b:int)->int:
    """Multiplies two numbers,Use this for math operations"""
    return a*b

In [15]:
@tool
def get_employee_email(name:str)->str:
    """Get employee email by name,Use this for HR operations"""
    #Mock Database
    db ={"John Doe": "john.doe@company.com", "Jane Smith": "jane.smith@company.com"}
    return db.get(name, "Employee not found")

In [16]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

In [17]:
class TicketInput(BaseModel):
    issue_type: str = Field(description="Type of the issue, e.g., 'Software', 'Hardware', 'Network'")
    priority: int = Field(description="Priority of the issue(1-5), e.g., '1-Low', '2-Medium', '3-High', '4-Critical', '5-Blocker'")
    description: str = Field(description="Detailed description of the issue/problem")


In [18]:
def create_it_ticket(issue_type: str, priority: int, description: str) -> str:
    """Creates an IT support ticket based on the provided details."""
    # Mock ticket creation logic
    ticket_id = f"TICKET-{issue_type[:3].upper()}-{priority}-{len(description)}"
    return f"Ticket created successfully! Ticket ID: {ticket_id}"

In [19]:
ticket_tool=StructuredTool.from_function(
    func=create_it_ticket,
    name="ticket_tool",
    description="Creates an IT support ticket based on the provided details.",
    args_schema=TicketInput
)

In [20]:
print(ticket_tool.invoke({"issue_type": "Software", "priority": 3, "description": "Application is not responding"}))

Ticket created successfully! Ticket ID: TICKET-SOF-3-29


In [21]:
tools=[multiply,get_employee_email,ticket_tool]

llm = ChatOllama(model="gpt-oss:120b-cloud")

llm_with_tools = llm.bind_tools(tools)

query= "My laptop screen is broken, It's urgent! and what is 15*17 and who is the prime minister of India?"

response = llm_with_tools.invoke(query)

print("------ RAW LLM Response--------:")
print(f" Content: {response.content} ")
print(f" Tool Calls: {response.tool_calls} ")



------ RAW LLM Response--------:
 Content:  
 Tool Calls: [{'name': 'multiply', 'args': {'a': 15, 'b': 17}, 'id': '881025fc-9919-4018-985e-499764841d3a', 'type': 'tool_call'}] 


In [22]:
from langchain_core.messages import HumanMessage,ToolMessage,AIMessage

messages = [ HumanMessage(content="Who is the prime minister of India? and multiply 15 by 17?My laptop screen is broken, It's urgent! ") ]


ai_msg = llm_with_tools.invoke(messages)

print(ai_msg)

messages.append(ai_msg)

print(ai_msg.tool_calls)




content='' additional_kwargs={} response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-07-20T06:19:39.003990277Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3259121924, 'load_duration': None, 'prompt_eval_count': 291, 'prompt_eval_duration': None, 'eval_count': 191, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'} id='lc_run--019f7e2e-07c3-7473-b57f-d345bc579cf5-0' tool_calls=[{'name': 'ticket_tool', 'args': {'description': 'Laptop screen is broken, urgent. Needs immediate replacement or repair.', 'issue_type': 'Hardware', 'priority': 4}, 'id': 'fb0ebd18-7583-41df-bf7c-6b57fb274c46', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 291, 'output_tokens': 191, 'total_tokens': 482}
[{'name': 'ticket_tool', 'args': {'description': 'Laptop screen is broken, urgent. Needs immediate replacement or repair.', 'issue_type': 'Hardware', 'priority': 4}, 'id': 'fb0ebd18-7583-41df-bf7c-6b57fb274c46', '

In [23]:
tool_map = {t.name: t for t in tools}

# gpt-oss via Ollama returns tool calls one at a time rather than all at once,
# so keep invoking the LLM and running whatever it asks for until it stops calling tools.
while ai_msg.tool_calls:
    for tool_call in ai_msg.tool_calls:
        selected_tool = tool_map[tool_call["name"]]

        tool_output = selected_tool.invoke(tool_call["args"])

        tool_msg = ToolMessage(tool_output, tool_call_id=tool_call["id"])

        messages.append(tool_msg)

        print(f"Executed {tool_call['name']} with output: {tool_output}")

    ai_msg = llm_with_tools.invoke(messages)
    messages.append(ai_msg)


Executed ticket_tool with output: Ticket created successfully! Ticket ID: TICKET-HAR-4-71


In [24]:
print(f"Messages: {messages}")
print(f"Final Response: {ai_msg.content}")

Messages: [HumanMessage(content="Who is the prime minister of India? and multiply 15 by 17?My laptop screen is broken, It's urgent! ", additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-07-20T06:19:39.003990277Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3259121924, 'load_duration': None, 'prompt_eval_count': 291, 'prompt_eval_duration': None, 'eval_count': 191, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--019f7e2e-07c3-7473-b57f-d345bc579cf5-0', tool_calls=[{'name': 'ticket_tool', 'args': {'description': 'Laptop screen is broken, urgent. Needs immediate replacement or repair.', 'issue_type': 'Hardware', 'priority': 4}, 'id': 'fb0ebd18-7583-41df-bf7c-6b57fb274c46', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 291, 'output_tokens': 191, 'total_tokens': 482}), ToolMessage(con